In [45]:
import pandas as pd
import numpy as np

In [46]:
df_antigo = pd.read_csv('../../dados/raw/reservas.csv', sep=';')
df_novo = pd.read_csv('../../dados/raw/reservas_novas.csv', sep=';')

In [47]:
df = pd.concat(
    [df_antigo, df_novo],
    ignore_index=True
)
df

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
0,1,5,1,3,2.0,2024-04-28,2024-04-30,2,1,4.0,confirmada,Cartão de Crédito
1,2,5,1,261,4.0,2023-09-28,2023-10-01,3,2,1.0,concluída,PIX
2,3,10,4,49,4.0,2023-08-01,2023-08-06,5,1,9.0,concluída,Cartão de Crédito
3,4,6,2,23,2.0,2023-02-18,2023-02-21,3,1,9.0,cancelada,PIX
4,5,7,1,42,1.0,2023-01-12,2023-01-19,7,1,6.0,confirmada,Transferência
...,...,...,...,...,...,...,...,...,...,...,...,...
4659,3198,7,5,268,1.0,2026-08-05,2026-08-07,2,2,9.0,confirmada,Cartão de Débito
4660,3490,9,3,48,1.0,2026-05-24,2026-05-26,2,3,8.0,CONFIRMADA,Transferência
4661,4579,13,6,306,4.0,2026-04-16,2026-04-18,2,4,NaN,confirmada,Dinheiro
4662,3882,11,1,100,4.0,2025-01-03,2025-01-06,3,1,8.0,CONFIRMADA,CD


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4664 entries, 0 to 4663
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_reserva         4664 non-null   int64  
 1   id_unidade         4664 non-null   int64  
 2   id_tipo_quarto     4664 non-null   int64  
 3   id_cliente         4664 non-null   int64  
 4   id_canal           4455 non-null   float64
 5   data_checkin       4664 non-null   str    
 6   data_checkout      4664 non-null   str    
 7   qtd_diarias        4664 non-null   int64  
 8   num_hospedes       4664 non-null   int64  
 9   avaliacao_hospede  4018 non-null   float64
 10  status_reserva     4664 non-null   str    
 11  forma_pagamento    4664 non-null   str    
dtypes: float64(2), int64(6), str(4)
memory usage: 437.4 KB


### Tratamento id_canal

In [49]:
df['id_canal'] = df['id_canal'].astype('Int64')

### Tratamento datas:

In [50]:
df['data_checkin'] = pd.to_datetime(
    df['data_checkin'],
    yearfirst=True
)
df['data_checkout'] = pd.to_datetime(
    df['data_checkout'],
    yearfirst=True
)

### Tratamento de textos:


#### Status reserva:

In [51]:
df['status_reserva'] = df['status_reserva'].str.strip().str.title()

In [52]:
map_status_reserva = {
    'Concluída': 'Confirmada',
    'Concluida': 'Confirmada',
    'Conf.': 'Confirmada',
    'Cancel.': 'Cancelada',
}
df['status_reserva'] = df['status_reserva'].map(map_status_reserva).fillna(df['status_reserva'])

In [53]:
df['status_reserva'].value_counts().reset_index()

,status_reserva,count
0,Confirmada,3767
1,Cancelada,655
2,No-Show,242


#### Forma pagamento:

In [54]:
df['forma_pagamento'].value_counts().reset_index()

,forma_pagamento,count
0,Dinheiro,909
1,Cartão de Débito,866
2,PIX,853
3,Transferência,851
4,Cartão de Crédito,823
5,pix,36
6,Pix,34
7,dinheiro,34
8,CD,34
9,C. Crédito,33


In [55]:
df['forma_pagamento'] = df['forma_pagamento'].str.strip().str.title()

In [56]:
map_pagamento = {
    'Cash': 'Dinheiro',
    # Debito
    'Cd': 'Cartão De Débito',
    'Déb': 'Cartão De Débito',
    'Cartao Debito': 'Cartão De Débito',
    # Transferencia
    'Ted': 'Transferência',
    'Transferencia': 'Transferência',
    # Credito
    'Cc': 'Cartão De Crédito',
    'Cred': 'Cartão De Crédito',
    'C. Crédito': 'Cartão De Crédito',
    'Cartao Credito': 'Cartão De Crédito',
    'Cartao Credito': 'Cartão De Crédito',
}
df['forma_pagamento'] = df['forma_pagamento'].map(map_pagamento).fillna(df['forma_pagamento'])

In [57]:
df['forma_pagamento'].value_counts().reset_index()

,forma_pagamento,count
0,Dinheiro,981
1,Cartão De Débito,943
2,Pix,923
3,Cartão De Crédito,913
4,Transferência,904


### Duplicatas:

In [58]:
df[df.duplicated(subset=['id_unidade', 'id_tipo_quarto', 'id_cliente', 'id_canal', 'data_checkin', 'data_checkout', 'qtd_diarias', 'num_hospedes', 'avaliacao_hospede', 'status_reserva', 'forma_pagamento'], keep=False)].sort_values(by='id_cliente')

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
585,586,5,2,2,2,2024-06-15,2024-06-17,2,1,3.0,Confirmada,Cartão De Crédito
2218,2219,5,2,2,2,2024-06-15,2024-06-17,2,1,3.0,Confirmada,Cartão De Crédito
3693,2851,4,4,6,3,2025-07-10,2025-07-14,4,2,NaN,Confirmada,Cartão De Débito
4473,2851,4,4,6,3,2025-07-10,2025-07-14,4,2,NaN,Confirmada,Cartão De Débito
1342,1343,9,1,13,1,2024-01-26,2024-02-01,6,1,9.0,Confirmada,Dinheiro
...,...,...,...,...,...,...,...,...,...,...,...,...
4413,4463,5,1,278,3,2025-07-19,2025-07-20,1,-1,2.0,Cancelada,Cartão De Crédito
1954,1955,7,3,290,4,2023-12-16,2023-12-17,1,3,9.0,Confirmada,Dinheiro
994,995,7,3,290,4,2023-12-16,2023-12-17,1,3,9.0,Confirmada,Dinheiro
2845,4560,13,7,302,2,2025-07-20,2025-07-23,3,3,8.0,Confirmada,Pix


### Nulos e analise final:

In [59]:
df.isnull().sum().reset_index()

,index,0
0,id_reserva,0
1,id_unidade,0
2,id_tipo_quarto,0
3,id_cliente,0
4,id_canal,209
5,data_checkin,0
6,data_checkout,0
7,qtd_diarias,0
8,num_hospedes,0
9,avaliacao_hospede,646


In [60]:
df[df.isnull() 
.any(axis=1)]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
7,8,4,4,33,3,2023-08-26,2023-09-02,7,2,NaN,No-Show,Dinheiro
10,11,8,3,189,<NA>,2024-10-08,2024-10-13,5,1,8.0,Confirmada,Transferência
12,13,3,3,184,<NA>,2024-02-01,2024-02-02,1,1,2.0,Confirmada,Pix
13,14,4,3,68,3,2024-03-07,2024-03-11,4,3,NaN,Cancelada,Dinheiro
14,15,6,2,173,1,2024-07-20,2024-07-23,3,1,NaN,Cancelada,Transferência
...,...,...,...,...,...,...,...,...,...,...,...,...
4651,4172,3,1,102,2,2025-04-01,2025-04-02,1,4,NaN,Confirmada,Dinheiro
4652,4462,5,1,166,<NA>,2025-04-28,2025-04-29,1,3,3.0,Confirmada,Dinheiro
4654,3526,9,5,116,<NA>,2026-03-03,2026-03-07,4,2,7.0,Confirmada,Cartão De Crédito
4658,4440,5,1,36,<NA>,2026-02-05,2026-02-07,2,1,1.0,Confirmada,Dinheiro


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4664 entries, 0 to 4663
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_reserva         4664 non-null   int64         
 1   id_unidade         4664 non-null   int64         
 2   id_tipo_quarto     4664 non-null   int64         
 3   id_cliente         4664 non-null   int64         
 4   id_canal           4455 non-null   Int64         
 5   data_checkin       4664 non-null   datetime64[us]
 6   data_checkout      4664 non-null   datetime64[us]
 7   qtd_diarias        4664 non-null   int64         
 8   num_hospedes       4664 non-null   int64         
 9   avaliacao_hospede  4018 non-null   float64       
 10  status_reserva     4664 non-null   str           
 11  forma_pagamento    4664 non-null   str           
dtypes: Int64(1), datetime64[us](2), float64(1), int64(6), str(2)
memory usage: 441.9 KB


### Adições e alterações:

In [62]:
def corrigir_valores(dados, coluna):
    for id_reser, correc  in dados.items():
        df.loc[df['id_reserva'] == id_reser, coluna] = correc

#### qtd_diarias:

In [63]:
correcoes_diarias = {
    728: 7,
    955: 4,
    1134: 3,
    1152: 2,
    1148: 6,
    1581: 4,
    2312: 4,
    2384: 2,
    3472: 1,
    3505: 1,
    3838: 1,
    4048: 1,
    2509: 1,
    4030: 1,
}

corrigir_valores(correcoes_diarias, 'qtd_diarias')

#### avaliac_hosp

In [64]:
correcoes_avaliac = {
    37: 1,
    902: 1,
    2073: 1,
    933: 3,
    1718: 3,
    2167: 3,
    3541: 1,
    2874: 1,
    4127: 1,
    4088: 3,
}

corrigir_valores(correcoes_avaliac, 'avaliacao_hospede')
# Tratamento para nulos: 
# Nulos são tratados como 0, já que são considerados 'não avaliados'
df['avaliacao_hospede'] = df['avaliacao_hospede'].fillna(0)


#### num_hosp

In [65]:
correcoes_hosp = {
    568: 1,
    675: 2,
    903: 1,
    1201: 2,
    1883: 2,
    2427: 2,
    4463: 1,
    4506: 2,
    4040: 2,
    2587: 2,
    4463: 1,
    4506: 2,
    4040: 2,
    2687: 2,
}

corrigir_valores(correcoes_hosp, 'num_hospedes')

#### Canal:

In [66]:
df['id_canal'] = df['id_canal'].fillna(0)

### Duplicatas:

In [67]:
df[df.duplicated(subset=['id_reserva',	'id_unidade',	'id_tipo_quarto',	'id_cliente', 'data_checkin', 'data_checkout', 'qtd_diarias', 'num_hospedes', 'avaliacao_hospede', 'status_reserva', 'forma_pagamento'], keep=False)].sort_values(by='id_cliente')

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
4473,2851,4,4,6,3,2025-07-10,2025-07-14,4,2,0.0,Confirmada,Cartão De Débito
3693,2851,4,4,6,3,2025-07-10,2025-07-14,4,2,0.0,Confirmada,Cartão De Débito
3202,2747,2,1,28,3,2026-07-05,2026-07-08,3,2,7.0,Cancelada,Pix
3940,2747,2,1,28,3,2026-07-05,2026-07-08,3,2,7.0,Cancelada,Pix
3760,2486,1,1,93,2,2025-09-26,2025-09-30,4,2,8.0,Confirmada,Cartão De Crédito
3544,2486,1,1,93,2,2025-09-26,2025-09-30,4,2,8.0,Confirmada,Cartão De Crédito
3382,4172,3,1,102,2,2025-04-01,2025-04-02,1,4,0.0,Confirmada,Dinheiro
4651,4172,3,1,102,2,2025-04-01,2025-04-02,1,4,0.0,Confirmada,Dinheiro
2583,4558,13,8,110,1,2025-07-20,2025-07-22,2,2,8.0,Confirmada,Cartão De Débito
2515,4558,13,8,110,1,2025-07-20,2025-07-22,2,2,8.0,Confirmada,Cartão De Débito


In [68]:
df = df.drop_duplicates(subset=['id_reserva', 'id_unidade','id_tipo_quarto', 'id_cliente', 'id_canal', 'data_checkin', 'data_checkout', 'qtd_diarias', 'num_hospedes', 'avaliacao_hospede', 'status_reserva', 'forma_pagamento'], keep='first')

In [69]:
df[df.isnull() 
.any(axis=1)]

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento


In [70]:
df.head(20)

,id_reserva,id_unidade,id_tipo_quarto,id_cliente,id_canal,data_checkin,data_checkout,qtd_diarias,num_hospedes,avaliacao_hospede,status_reserva,forma_pagamento
0,1,5,1,3,2,2024-04-28,2024-04-30,2,1,4.0,Confirmada,Cartão De Crédito
1,2,5,1,261,4,2023-09-28,2023-10-01,3,2,1.0,Confirmada,Pix
2,3,10,4,49,4,2023-08-01,2023-08-06,5,1,9.0,Confirmada,Cartão De Crédito
3,4,6,2,23,2,2023-02-18,2023-02-21,3,1,9.0,Cancelada,Pix
4,5,7,1,42,1,2023-01-12,2023-01-19,7,1,6.0,Confirmada,Transferência
5,6,4,3,179,3,2024-04-28,2024-05-03,5,3,10.0,Confirmada,Transferência
6,7,10,2,144,2,2023-04-19,2023-04-24,5,1,9.0,Confirmada,Dinheiro
7,8,4,4,33,3,2023-08-26,2023-09-02,7,2,0.0,No-Show,Dinheiro
8,9,2,3,140,2,2024-04-04,2024-04-08,4,3,6.0,Confirmada,Dinheiro
9,10,5,1,48,2,2023-07-02,2023-07-04,2,1,2.0,Confirmada,Pix


In [71]:
df.to_csv('../../dados/clean/reservas.csv', sep=';', encoding='UTF-8', decimal='.')